# 1.6 - Tropical (eastward) sub-clustering (two-stage)

The eastward retroflection / tropical pathway is a small share of the particles,
so the global k-means (notebook 1.5) gives it only a few clusters - k-means
spends clusters where the data is dense. **Two stages:** pick out the
eastward particles first, then cluster only them with their own K.

**Selection by direction (not by end point).** For each particle, take its exit
point at `GATE_R` km (default 1000 km) and the compass direction from its own
2.5N crossing point to that exit (0 = N, 90 = E, -90 = W). Particles whose
direction is inside `BEARING_RANGE` are kept. Almost every particle reaches
1000 km (~97-99%), so this loses far fewer particles than a rule on the
outermost radius: eastward particles often end their track before 2000 km
(`partial`), and are kept here (`FIT_STATUS`), with their last position
standing in for the radii they did not reach.

In a first test most of the eastward subset was `partial` (few reach 1500 km,
almost none 2000 km), so their 1500 / 2000 km features are mostly the end of
the track. The check after the selection flags such radii; setting their
weight to 0 clusters on the radii the subset really reaches.

**One particle as an example:** it crosses 2.5N at 49W, turns back east at
about 7N and is 1000 km from its crossing point near 8N 41W. The direction
crossing -> exit is about 60 (north-east), inside the default (0, 180), so it is
kept. A particle that goes along the shelf to the north-west exits at about
-45 and is left out.

Workflow, as in 1.5: tune the settings, run top to bottom, give each
sub-cluster a letter in `ASSIGN`, set `SAVE_MODEL = True` and re-run the save
cells to write the model and the per-particle sub-labels.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))   # project root: config.py, pipeline.py
import numpy as np
import pandas as pd
import config as C
import pipeline as P
print("project root:", C.PROJECT_ROOT)
print(f"crossing: {C.LAT_CROSS} N | radii (km): {C.RADII_KM}")

import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [ ]:
K_TROPICAL = 20                      # clusters for the eastward subset only
GATE_R = 1000                        # radius whose exit direction selects (one of C.RADII_KM)
BEARING_RANGE = (0, 180)             # kept directions, from-to in deg clockwise from N (0 N, 90 E, -90 W)
FIT_STATUS = ["complete", "partial"] # statuses kept (partial: track ended before a larger radius)
SAVE_MODEL = False                   # True -> write model + sub-labels (save cells)

In [ ]:
# per-radius weights tuned HERE for the subset (None -> config.RADIUS_WEIGHTS).
# 0 switches a radius off: use it for radii most of the subset never reaches
# (their value is only the last position of the track - see the check below).
RADIUS_WEIGHTS = {200: 1.0, 500: 1.0, 1000: 1.0, 1500: 1.0, 2000: 1.0}

assert GATE_R in C.RADII_KM, f"GATE_R must be one of {C.RADII_KM}"
_wr = RADIUS_WEIGHTS if RADIUS_WEIGHTS is not None else C.RADIUS_WEIGHTS
W = P.feature_weight_vector(_wr)
print(f"K_TROPICAL: {K_TROPICAL} | gate: direction at {GATE_R} km in {BEARING_RANGE} "
      f"| statuses: {FIT_STATUS} | weights: {_wr}")

## Select the eastward subset

The direction is computed from the features file (`cross_lat/lon` and the exit
point at `GATE_R`), so no Zarr store is read. The histogram shows all particles
that reached `GATE_R`, with the kept range shaded - use it to set
`BEARING_RANGE`.

In [ ]:
features = pd.read_parquet(C.FEATURES_FILE)
cand = features[features.status.isin(FIT_STATUS)
                & features[P.days_col(GATE_R)].notna()].copy()     # really reached GATE_R
cand["bearing"] = P.bearing_deg(cand.cross_lat, cand.cross_lon,
                                cand[P.lat_col(GATE_R)], cand[P.lon_col(GATE_R)])
keep = cand.bearing.between(*BEARING_RANGE)
trop = cand[keep].reset_index(drop=True)

n_lab = features.status.isin(["complete", "partial"]).sum()
print(f"reached {GATE_R} km: {len(cand):,} | kept (direction in {BEARING_RANGE}): "
      f"{len(trop):,} = {100*len(trop)/len(cand):.2f}% of them, "
      f"{100*len(trop)/n_lab:.2f}% of all labelled particles")
print("status in the subset:", trop.status.value_counts().to_dict())
for r in C.RADII_KM:
    got = 100 * trop[P.days_col(r)].notna().mean()
    warn = (f"   <-- mostly last-position stand-ins; consider RADIUS_WEIGHTS[{r}] = 0"
            if got < 50 and _wr[r] > 0 else "")
    print(f"  reached {r:5d} km: {got:5.1f}% of the subset (weight {_wr[r]}){warn}")

fig, ax = plt.subplots(figsize=(9, 3.2))
ax.hist(cand.bearing, bins=np.arange(-180, 181, 5), color="0.5")
ax.axvspan(*BEARING_RANGE, color="green", alpha=.15, label="kept")
ax.set_yscale("log")
ax.set_xticks([-180, -135, -90, -45, 0, 45, 90, 135, 180])
ax.set_xticklabels(["S", "SW", "W", "NW", "N", "NE", "E", "SE", "S"])
ax.set_xlabel(f"direction crossing point -> {GATE_R} km exit"); ax.set_ylabel("particles (log)")
ax.legend(frameon=False); fig.tight_layout(); plt.show()

## Cheap coastline and the gate drawn on the maps
The `GATE_R` circle around the mean crossing point of the subset, green over
the kept directions (each particle really uses its own crossing point).

In [ ]:
import matplotlib.patches as mpatches

def draw_refs(ax, **kw):
    # reference lines from the day-based notebooks + the 2.5N starting line
    for x in (-70, -60, -40):
        ax.axvline(x, lw=.5, **kw)
    ax.axhline(20, lw=.5, **kw)
    ax.axhline(C.LAT_CROSS, lw=.6, ls="--", color="k")
    ax.add_patch(mpatches.Rectangle(xy=(-55, 5), width=10, height=5, lw=1,
                                    edgecolor="green", facecolor="none"))

import cartopy.feature as cfeature
lon_lim = (C.DOMAIN["lon_min"], C.DOMAIN["lon_max"])
lat_lim = (C.DOMAIN["lat_min"], C.DOMAIN["lat_max"])

_coast = []
for geom in cfeature.COASTLINE.with_scale("110m").geometries():
    for line in getattr(geom, "geoms", [geom]):
        _coast.append((np.asarray(line.xy[0]), np.asarray(line.xy[1])))

def draw_coast(ax):
    for x, y in _coast:
        ax.plot(x, y, "-", color="0.75", lw=.4, zorder=0)
    ax.set_xlim(*lon_lim); ax.set_ylim(*lat_lim)

_gla, _glo = P.circle_latlon(C.LAT_CROSS, float(trop.cross_lon.mean()), GATE_R)   # point i = bearing i deg
_gb = np.arange(_gla.size); _gb = np.where(_gb > 180, _gb - 360, _gb)
_gin = (_gb >= BEARING_RANGE[0]) & (_gb <= BEARING_RANGE[1])

def draw_gate(ax):
    ax.plot(_glo, _gla, "-", color="0.6", lw=.6, zorder=1)
    ax.plot(np.where(_gin, _glo, np.nan), np.where(_gin, _gla, np.nan), "-",
            color="green", lw=1.6, zorder=1)
print("coastline segments:", len(_coast))

In [ ]:
# where the kept / left-out particles exit the GATE_R circle (random 20k)
s = cand.sample(min(20000, len(cand)), random_state=C.RANDOM_STATE)
k = s.bearing.between(*BEARING_RANGE)
fig, ax = plt.subplots(figsize=(9, 5))
draw_coast(ax); draw_refs(ax); draw_gate(ax)
ax.scatter(s.loc[~k, P.lon_col(GATE_R)], s.loc[~k, P.lat_col(GATE_R)], s=2, color="0.6",
           label="left out")
ax.scatter(s.loc[k, P.lon_col(GATE_R)], s.loc[k, P.lat_col(GATE_R)], s=4, color="green",
           label="kept")
ax.legend(markerscale=4, loc="lower left")
ax.set_title(f"exit points at {GATE_R} km (random {len(s):,}); green arc = kept directions")
fig.tight_layout(); plt.show()

## Fit k-means on the subset
Small subset -> plain `KMeans`. Standardize each column on the subset, apply the
radius weights, fit.

In [ ]:
K = K_TROPICAL
X_raw  = P.build_feature_matrix(trop)
scaler = StandardScaler().fit(X_raw)
X_sub  = scaler.transform(X_raw) * W

km = KMeans(n_clusters=K, random_state=C.RANDOM_STATE, n_init=10)
labels = km.fit_predict(X_sub)
sizes  = np.bincount(labels, minlength=K)

# centroids in degrees = mean member position per radius (same as undoing the
# scaling when every weight > 0, and still defined for radii with weight 0)
cent_deg  = pd.DataFrame(X_raw).groupby(labels).mean().reindex(range(K)).to_numpy()
start_lon = trop.groupby(labels).cross_lon.mean().reindex(range(K)).to_numpy()
def centroid_track(c):
    return (np.r_[start_lon[c], cent_deg[c, 1::2]],
            np.r_[C.LAT_CROSS, cent_deg[c, 0::2]])
print(f"fit k={K} on {len(trop):,} eastward particles | sizes {sizes.min()}..{sizes.max()}")

In [ ]:
# Persist THIS fit (only when SAVE_MODEL is True).  The subsample labels keep
# store / row / traj, so every particle can be found again.
import joblib, json
MODEL_DIR = C.MODELS_DIR / f"tropical_1.6_k{K}"
if SAVE_MODEL:
    MODEL_DIR.mkdir(parents=True, exist_ok=True)
    joblib.dump(km, MODEL_DIR / f"kmeans_k{K}.pkl")
    joblib.dump(scaler, MODEL_DIR / "scaler.pkl")
    np.save(MODEL_DIR / "weights_W.npy", W)
    np.save(MODEL_DIR / "centroids_deg.npy", cent_deg)
    (MODEL_DIR / "fit_metadata.json").write_text(json.dumps(dict(
        source_notebook="1.6_Tropical_subclustering.ipynb", K=K, LAT_CROSS=C.LAT_CROSS,
        GATE_R=GATE_R, BEARING_RANGE=list(BEARING_RANGE), FIT_STATUS=FIT_STATUS,
        RADII_KM=C.RADII_KM, RADIUS_WEIGHTS={str(k): v for k, v in _wr.items()},
        RANDOM_STATE=C.RANDOM_STATE, n_fitted=int(len(trop)),
        cluster_sizes=sizes.tolist()), indent=2))
    print("saved model ->", MODEL_DIR)
else:
    print("SAVE_MODEL is False -> nothing written")

## Read a few full member trajectories per sub-cluster
By `store` / `row`, checked against `traj`; each store opened once.

In [ ]:
MEMBERS_PER_PANEL = 20
N_IO_THREADS = int(os.environ.get("SLURM_CPUS_PER_TASK", 16))

memdf = trop[["trajectory_id", "store", "row", "traj", "cross_obs"]].assign(lab=labels)
pick = pd.concat([d.sample(min(MEMBERS_PER_PANEL, len(d)), random_state=C.RANDOM_STATE)
                  for _, d in memdf.groupby("lab")], ignore_index=True)
paths = P.read_paths(pick, n_jobs=N_IO_THREADS)          # trajectory_id -> (lon, lat)
cross_obs = dict(zip(pick.trajectory_id, pick.cross_obs))
members = {c: pick.loc[pick.lab == c, "trajectory_id"].to_numpy() for c in range(K)}
print(f"read {len(paths):,} full trajectories from {pick.store.nunique()} stores")

## The grid - one panel per sub-cluster
Orange = centroid (mean crossing point -> exit at each radius), blue = member
tracks after the crossing, grey = before; green arc = the kept directions.

In [ ]:
NCOL = 5

def draw_cluster(ax, c):
    draw_coast(ax); draw_gate(ax)
    for tid in members[c]:
        lo, la = paths[int(tid)]
        k0 = cross_obs[tid]
        ax.plot(lo[:k0 + 1], la[:k0 + 1], "-", color="0.6", lw=.4, alpha=.4)
        m = ~(np.isnan(lo[k0:]) | np.isnan(la[k0:]))
        ax.plot(lo[k0:][m], la[k0:][m], "-", color="blue", lw=.4, alpha=.35)
    clo, cla = centroid_track(c)
    ax.plot(clo, cla, ".-", color="darkorange", lw=2, zorder=3)
    ax.plot(clo[-1], cla[-1], "o", color="darkorange", mec="k", ms=6, zorder=4)

nrow = int(np.ceil(K / NCOL))
fig, axes = plt.subplots(nrow, NCOL, figsize=(3.2 * NCOL, 2.5 * nrow),
                         squeeze=False, sharex=True, sharey=True)
for c in range(K):
    ax = axes[c // NCOL][c % NCOL]
    draw_cluster(ax, c)
    ax.set_title(f"sub {c}  (n={sizes[c]:,}, partial {100*(trop.status[labels == c] == 'partial').mean():.0f}%)",
                 fontsize=8)
for j in range(K, nrow * NCOL):
    axes[j // NCOL][j % NCOL].axis("off")
fig.suptitle(f"Eastward sub-clusters k={K} (orange = centroid, blue = member tracks after crossing)", y=1.0)
fig.tight_layout(); plt.show()

## All sub-clusters on one map

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))
draw_coast(ax); draw_refs(ax); draw_gate(ax)
allcmap = plt.get_cmap("tab20", K)
for c in range(K):
    lo, la = centroid_track(c)
    ax.plot(lo, la, "-", color=allcmap(c), lw=1.4, alpha=.85, zorder=2)
    ax.text(lo[-1], la[-1], str(c), fontsize=8, ha="center", va="center", zorder=4)
ax.set_title(f"All {K} eastward sub-cluster centroids (crossing -> {C.RADII_KM[-1]} km)")
fig.tight_layout(); plt.show()

## Assign each sub-cluster a group letter
`ASSIGN[cluster] = "letter"`; same letter => merged. Starts all separate.

In [ ]:
import string
def lbl(c):
    return string.ascii_uppercase[c] if c < 26 else f"g{c}"
ASSIGN = {c: lbl(c) for c in range(K)}

# --- merge by hand, e.g.: ---
# for c in [0, 3]:  ASSIGN[c] = "A"

ASSIGN

## Preview the merged groups

In [ ]:
letters = sorted(set(ASSIGN.values()))
NCOLg = min(4, len(letters))
nrowg = int(np.ceil(len(letters) / NCOLg))
cmap  = plt.get_cmap("tab20", K)
fig, axes = plt.subplots(nrowg, NCOLg, figsize=(3.4 * NCOLg, 2.7 * nrowg),
                         squeeze=False, sharex=True, sharey=True)
for i, L in enumerate(letters):
    ax = axes[i // NCOLg][i % NCOLg]
    draw_coast(ax); draw_gate(ax)
    mem = [c for c in range(K) if ASSIGN[c] == L]
    for c in mem:
        lo, la = centroid_track(c)
        ax.plot(lo, la, "-", color=cmap(c), lw=1.6)
        ax.plot(lo[-1], la[-1], "o", color=cmap(c), mec="k", ms=5)
    ax.set_title(f"group {L}  ({len(mem)} sub-clusters)", fontsize=9)
for j in range(len(letters), nrowg * NCOLg):
    axes[j // NCOLg][j % NCOLg].axis("off")
fig.suptitle("Merged eastward groups (centroids)", y=1.0)
fig.tight_layout(); plt.show()

## Export
Prints the grouping; with `SAVE_MODEL = True` also writes one row per subset
particle (`store` / `row` / `traj`, sub-cluster, group) to
`data/tropical_1.6_labels_k<K>.parquet`.

In [ ]:
name2id     = {L: i for i, L in enumerate(letters)}
SUB_MAP     = {c: name2id[ASSIGN[c]] for c in range(K)}
SUB_NAMES   = {i: L for L, i in name2id.items()}

print(f"# gate: direction at {GATE_R} km in {BEARING_RANGE}, statuses {FIT_STATUS} | weights: {_wr}")
print("SUB_MAP =", SUB_MAP)
print("SUB_NAMES =", SUB_NAMES)
print()
for L in letters:
    mem = [c for c in range(K) if ASSIGN[c] == L]
    print(f"  {L}: {mem}  (n={sum(sizes[c] for c in mem):,})")

if SAVE_MODEL:
    out = trop[["trajectory_id", "store", "row", "traj", "status", "bearing"]].assign(
        sub_label=labels.astype(np.int32))
    out["sub_group"] = out.sub_label.map(SUB_MAP).astype(np.int32)
    out["sub_name"] = out.sub_group.map(SUB_NAMES)
    OUT = C.DATA_DIR / f"tropical_1.6_labels_k{K}.parquet"
    out.to_parquet(OUT, index=False)
    print("\nsaved", out.shape, "->", OUT)